# Atelier RAG — Interrogez vos propres documents

> Journée 3 · LLMs · *Retrieval-Augmented Generation*

Un LLM ne connaît **que** ce qu'il a vu pendant son entraînement. Il ne connaît ni vos
documents internes, ni les données postérieures à sa date de coupure. Pire : interrogé
hors de son savoir, il **hallucine** (invente une réponse plausible mais fausse).

Le **RAG** (*Retrieval-Augmented Generation*) résout ce problème sans ré-entraîner le modèle :
on **récupère** (retrieve) les passages pertinents d'une base documentaire, puis on les
**injecte** dans le prompt pour que le LLM réponde *en s'appuyant sur ces passages*.

**Fine-tuning vs RAG** : le fine-tuning modifie les *poids* du modèle (coûteux, fige la
connaissance) ; le RAG modifie le *contexte* à l'inférence (peu coûteux, connaissance
toujours à jour, sources traçables). En entreprise, le RAG est le premier réflexe.

## Le pipeline RAG

```
Documents -> (1) Decoupage en chunks -> (2) Embeddings -> Index vectoriel
                                                              |                         
Question utilisateur -> Embedding de la question -> (3) Recherche semantique (top-k)
                                                              |                     
                                        (4) Prompt augmente (question + chunks)
                                                              |                         
                                                 (5) LLM -> Reponse sourcée
```

Nous allons construire ce pipeline **étape par étape**, en repartant à chaque fois du
problème concret. Stack : `sentence-transformers` (embeddings) + un LLM local
(`ollama`, repli sur `flan-t5`). Aucune clé API requise.

In [ ]:
import numpy as np
import textwrap

# =====================================================================
# Base documentaire : un mini "manuel interne" d'une entreprise fictive.
# Un LLM ne peut PAS connaitre ces faits -> terrain ideal pour le RAG.
# =====================================================================
DOCUMENTS = {
    "conges": (
        "Politique de conges d'Ambient Robotics. Chaque employe dispose de 32 jours de "
        "conges payes par an. Les conges se demandent au minimum 15 jours a l'avance via "
        "l'outil interne Atlas. Les jours non pris sont perdus au 31 mai de l'annee suivante."
    ),
    "teletravail": (
        "Charte teletravail. Le teletravail est autorise jusqu'a 3 jours par semaine. "
        "Le mardi est un jour de presence obligatoire sur site pour toutes les equipes. "
        "Une indemnite forfaitaire de 2,60 euros par jour teletravaille est versee."
    ),
    "materiel": (
        "Equipement. Chaque ingenieur recoit un ordinateur portable, soit un MacBook Pro 14 "
        "pouces, soit un Dell XPS 15, au choix. Le renouvellement du materiel intervient tous "
        "les 4 ans. Les demandes de materiel supplementaire passent par le manager."
    ),
    "produit": (
        "Notre produit phare est le robot ARC-7, un robot logistique d'entrepot lance en "
        "mars 2024. L'ARC-7 embarque le firmware Helios et peut transporter jusqu'a 80 kg "
        "a une vitesse maximale de 1,8 m/s. Son autonomie est de 9 heures."
    ),
    "securite": (
        "Securite informatique. L'authentification a deux facteurs (2FA) est obligatoire "
        "pour tous les services. Les mots de passe doivent comporter au moins 14 caracteres "
        "et sont renouveles tous les 6 mois. Le partage de mot de passe est strictement interdit."
    ),
    "support": (
        "Support et astreinte. L'equipe support est joignable de 8h a 20h du lundi au samedi. "
        "En dehors de ces horaires, une astreinte gere uniquement les incidents critiques de "
        "niveau P1. Le delai de reponse garanti (SLA) pour un P1 est de 30 minutes."
    ),
}

print(f"{len(DOCUMENTS)} documents charges.")

## Étape 0 — Le problème : le LLM seul **hallucine**

Posons une question dont la réponse n'existe **que** dans nos documents internes.
Le modèle, n'ayant aucune connaissance d'Ambient Robotics, va soit inventer, soit avouer
son ignorance. Observons ce comportement *avant* d'ajouter le RAG.

In [ ]:
from transformers import pipeline

# On charge le modele UNE seule fois, au niveau module : le pipeline est ensuite
# reutilise a chaque appel de generate(). (Le recreer a chaque appel rechargerait
# plusieurs Go de poids depuis le disque a chaque question -> tres lent.)
pipe = pipeline("image-text-to-text", model="Qwen/Qwen3.5-0.8B")

def generate(question):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question}
            ]
        },
    ]
    return pipe(text=messages)


In [ ]:
question = "Combien de jours de conges payes ai-je par an chez Ambient Robotics ?"

# Generation SANS contexte : le modele n'a aucune source -> hallucination probable
print("Q :", question)

history = generate(question)
print("R (sans RAG) :", history[-1]['generated_text'][-1]['content'])

## Étape 1 — Découpage en *chunks*

On ne peut pas injecter toute la base dans le prompt (fenêtre de contexte limitée, coût,
bruit). On découpe donc les documents en **chunks** de taille bornée, avec un léger
**chevauchement** (*overlap*) pour ne pas couper une idée en deux.

**À compléter :** implémentez `chunk_text` qui découpe un texte en fenêtres de
`chunk_size` mots avec `overlap` mots de recouvrement.

In [ ]:
def chunk_text(text, chunk_size=40, overlap=10):
    """Decoupe `text` en chunks de `chunk_size` mots, avec `overlap` mots de chevauchement.
    Retourne une liste de chaines."""
    # TODO : decouper `text` en fenetres de `chunk_size` mots qui se chevauchent.
    #   1) words = text.split()
    #   2) avancez par pas de step = chunk_size - overlap
    #   3) pour chaque depart, prenez la fenetre words[start:start + chunk_size]
    #   4) ajoutez " ".join(window) a la liste des chunks (stop quand la fenetre est vide)
    #   5) retournez la liste des chunks
    ...


In [ ]:
# Construction de la base de chunks : on garde la trace du document source (pour citer)
chunks, chunk_sources = [], []
for doc_id, text in DOCUMENTS.items():
    for c in chunk_text(text, chunk_size=40, overlap=10):
        chunks.append(c)
        chunk_sources.append(doc_id)

print(f"{len(chunks)} chunks issus de {len(DOCUMENTS)} documents.")
print("Exemple de chunk [", chunk_sources[0], "]:\n", textwrap.fill(chunks[0], 90))

## Étape 2 — Embeddings

Un **embedding** transforme un texte en vecteur dense : deux textes de sens proche ont des
vecteurs proches. C'est ce qui permet la recherche *sémantique* (par le sens) plutôt que
par mots-clés exacts. On réutilise `sentence-transformers` (déjà vu en J3 sur Flipkart).

**À compléter :** encodez les `chunks` en une matrice `chunk_embeddings` de forme
`(n_chunks, dim)`.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")  # leger, multilingue (FR)

# TODO : encoder les `chunks` en une matrice (n_chunks, dim).
#   Utilisez embedder.encode(...) avec normalize_embeddings=True
#   (normaliser -> le produit scalaire devient directement la similarite cosinus).
chunk_embeddings = ...  # TODO
print("Forme des embeddings :", chunk_embeddings.shape)


## Étape 3 — Recherche sémantique (*retrieval*)

Pour une question, on calcule son embedding puis on récupère les `k` chunks les plus
proches au sens de la **similarité cosinus**. Comme les vecteurs sont normalisés, la
similarité cosinus se réduit à un simple produit scalaire.

**À compléter :** implémentez `retrieve(query, k)` qui renvoie les `k` meilleurs chunks
avec leur score et leur source.

In [ ]:
def retrieve(query, k=3):
    """Renvoie les k chunks les plus pertinents : liste de (score, source, chunk)."""
    # TODO :
    #   1) encodez la `query` : embedder.encode([query], normalize_embeddings=True)[0]
    #   2) scores = similarite cosinus = chunk_embeddings @ q_emb (vecteurs normalises)
    #   3) recuperez les indices des k meilleurs scores : np.argsort(scores)[::-1][:k]
    #   4) renvoyez une liste de tuples (float(score), chunk_sources[i], chunks[i])
    ...

for score, src, txt in retrieve("Combien de jours de conges ?", k=3):
    print(f"[{score:.3f}] ({src}) {textwrap.shorten(txt, 80)}")


In [ ]:
# Pourquoi "semantique" ? La question ne partage AUCUN mot avec le document pertinent
# ("teletravail", "site", "semaine"...). Une recherche par mots-cles le manquerait ;
# la recherche par embeddings le ramene bien dans le top-k grace au SENS.
q = "Puis-je travailler depuis chez moi ?"
print("Question :", q, "\n")
for score, src, txt in retrieve(q, k=3):
    print(f"[{score:.3f}] ({src}) {textwrap.shorten(txt, 90)}")

## Étape 4 — Augmentation du prompt

On assemble un prompt qui (a) fournit les chunks récupérés comme **contexte**, (b) pose la
question, et (c) **interdit au modèle de répondre hors de ce contexte** — c'est le garde-fou
anti-hallucination, essentiel en production.

**À compléter :** implémentez `build_rag_prompt(query, retrieved)`.

In [ ]:
def build_rag_prompt(query, retrieved):
    """Construit le prompt augmente a partir des chunks recuperes."""
    # TODO :
    #   1) assemblez le CONTEXTE a partir de `retrieved` (liste de (score, src, txt)),
    #      une ligne par chunk, ex. f"- ({src}) {txt}".
    #   2) construisez le `prompt` qui :
    #        - donne le role d'assistant interne,
    #        - impose de repondre UNIQUEMENT a partir du CONTEXTE,
    #        - impose de repondre exactement "Je ne sais pas" si l'info est absente (garde-fou),
    #        - demande de citer le document source entre parentheses,
    #        - insere ensuite le CONTEXTE puis la QUESTION.
    #   3) retournez le prompt (str).
    ...

print(build_rag_prompt(question, retrieve(question, k=3)))


## Étape 5 — Le pipeline complet

On enchaîne : `retrieve` ➜ `build_rag_prompt` ➜ `generate`. On renvoie aussi les sources
pour la **traçabilité** (un atout majeur du RAG face au fine-tuning).

**À compléter :** assemblez `rag_answer(query, k)`.

In [ ]:
SEUIL_PERTINENCE = 0.25   # en dessous, on considere qu'aucun passage n'est vraiment pertinent

def rag_answer(query, k=3, seuil=SEUIL_PERTINENCE):
    """Pipeline RAG complet. Renvoie (reponse, sources).

    Deux garde-fous complementaires :
      1. On ne garde que les chunks dont le score de similarite depasse `seuil`.
      2. Si AUCUN chunk ne passe le seuil, on repond "Je ne sais pas" sans meme
         appeler le LLM (economie de calcul + zero risque d'hallucination).
    On ne cite alors que les sources REELLEMENT utilisees (et non tout le top-k)."""
    # TODO : enchainez les briques du pipeline RAG, avec les deux garde-fous
    #   1) retrieved = retrieve(query, k=k)
    #   2) ne gardez que les chunks dont le score >= seuil (garde-fou 1)
    #   3) si aucun chunk ne passe le seuil -> renvoyez ("Je ne sais pas", []) SANS
    #      appeler le LLM (garde-fou 2)
    #   4) prompt = build_rag_prompt(query, <chunks retenus>)
    #   5) appelez generate(prompt) et extrayez le texte de la reponse
    #   6) sources = ensemble TRIE des documents sources des chunks RETENUS
    #   7) retournez (answer, sources)
    ...

ans, srcs = rag_answer(question)
print("Q :", question)
print("R (avec RAG) :", ans)
print("Sources :", srcs)


In [ ]:
# Batterie de questions : comparez la reponse RAG a la verite terrain
for q in [
    "Quelle est la charge maximale du robot ARC-7 ?",
    "Combien de caracteres minimum pour un mot de passe ?",
    "Quel jour la presence sur site est-elle obligatoire ?",
    "Quel est le SLA pour un incident critique ?",
]:
    ans, srcs = rag_answer(q, k=3)
    print("Q :", q)
    print("R :", ans, " | sources :", srcs)
    print("-" * 90)

## Étape 6 — Garde-fou : les questions hors périmètre

Un bon système RAG **sait dire qu'il ne sait pas**. Posons une question dont la réponse
n'existe dans aucun document. Deux filets de sécurité entrent alors en jeu :

1. **Avant le LLM** — si aucun chunk ne dépasse `SEUIL_PERTINENCE`, `rag_answer` répond
   « Je ne sais pas » sans même appeler le modèle (rapide, et zéro hallucination possible).
2. **Dans le LLM** — si des chunks passent le seuil mais ne contiennent pas la réponse, la
   consigne de l'étape 4 pousse le modèle à répondre « Je ne sais pas » plutôt qu'inventer.

In [ ]:
hors_scope = "Quel est le budget marketing 2025 d'Ambient Robotics ?"
ans, srcs = rag_answer(hors_scope, k=3)
print("Q :", hors_scope)
print("R :", ans)
print("Sources :", srcs)
# Si tous les scores de retrieval sont sous SEUIL_PERTINENCE, la reponse est "Je ne sais pas"
# et `sources` est vide : le LLM n'a meme pas ete appele (garde-fou 1). Sinon, c'est la
# consigne de l'etape 4 qui sert de filet (garde-fou 2).

## Exercices bonus (pour aller plus loin, façon *Building & Evaluating Advanced RAG*)

1. **Seuil de confiance (déjà en place)** — `rag_answer` refuse déjà de répondre sous
   `SEUIL_PERTINENCE` (0,25). Faites varier ce seuil : trop haut = des questions légitimes
   sont refusées ; trop bas = les questions hors périmètre repassent et le modèle hallucine.
2. **Sensibilité au chunking** — faites varier `chunk_size` ∈ {20, 40, 80} et `k` ∈ {1, 3, 5}.
   Quel est l'impact sur la pertinence ? Trop petit = contexte tronqué ; trop grand = bruit.
3. **MMR (diversité)** — au lieu du top-k brut, sélectionnez des chunks à la fois pertinents
   **et** peu redondants (*Maximal Marginal Relevance*).
4. **Vrai vector store** — remplacez la recherche numpy par **ChromaDB** (ci-dessous).
5. **Évaluation** — constituez 10 paires (question, réponse attendue) et mesurez un taux de
   réussite. C'est la base de toute démarche RAG sérieuse en production.

**À compléter (bonus 4) :** indexez les chunks dans ChromaDB et ré-implémentez `retrieve`.

In [ ]:
# BONUS 4 — ChromaDB (vector store persistant, utilise en production)
import chromadb

client = chromadb.Client()
# IMPORTANT : par defaut Chroma utilise la distance L2. Nos embeddings sont normalises
# et notre retriever numpy raisonne en cosinus -> on force l'espace cosinus pour que
# les scores soient coherents avec l'etape 3 (sinon `1 - dist` n'a aucun sens).
collection = client.get_or_create_collection(
    "ambient_robotics", metadata={"hnsw:space": "cosine"}
)

collection.add(
    documents=chunks,
    embeddings=[e.tolist() for e in chunk_embeddings],
    metadatas=[{"source": s} for s in chunk_sources],
    ids=[f"c{i}" for i in range(len(chunks))],
)

def retrieve_chroma(query, k=3):
    q_emb = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    res = collection.query(query_embeddings=[q_emb], n_results=k)
    out = []
    for txt, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        out.append((1 - dist, meta["source"], txt))  # espace cosinus : 1 - dist = similarite
    return out

for score, src, txt in retrieve_chroma("Combien de jours de conges ?", k=3):
    print(f"[{score:.3f}] ({src}) {textwrap.shorten(txt, 80)}")


## Récapitulatif

| Étape | Rôle | Brique |
|-------|------|--------|
| Chunking | borner le contexte | `chunk_text` |
| Embeddings | représenter le sens | `sentence-transformers` |
| Retrieval | trouver les passages utiles | similarité cosinus / ChromaDB |
| Prompt augmenté | ancrer + garde-fou | `build_rag_prompt` |
| Génération | répondre, sourcé | LLM local (Ollama / flan-t5) |

**À retenir** : le RAG apporte une connaissance *à jour*, *privée* et *traçable* sans
ré-entraîner le modèle. Les leviers de qualité ne sont pas le LLM seul mais surtout le
**retrieval** (chunking, embeddings, seuils, ré-ordonnancement) et l'**évaluation**.